<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 I. RAG Moderne avec LCEL 
</h1>



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 1 — Utiliser WordIngestion pour préparer les chunks
</h2>


In [1]:
from word_ingestion import WordIngestion

# ingestor personnalisé 
ingestor = WordIngestion(
    chunk_size= 500,
    chunk_overlap= 80
)

# Traiter un document word 
chunks = ingestor.process("../data/word_files/Document_RAG_Word.docx")

print("Nombre de chunks :", len(chunks))

# Aperçu des premiers chunks 
for i, c in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print("=" *80)
    print(c.page_content[:500], "\n")


Nombre de chunks : 7
CHUNK 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG
Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel.
1. Contexte général 

CHUNK 2
1. Contexte général
Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers techniques. Ces documents doivent être ingérés, nettoyés, découpés en chunks et indexés pour permettre à un modèle de répondre de façon cohérente aux requêtes des utilisateurs.
2. Challenges rencontrés 

CHUNK 



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 2 — Créer les embeddings + Vector Store (Chroma)
</h2>


In [2]:
# Import Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Initialisation du Vector Store
vector_store = Chroma(
    collection_name="rag_word_docs",
    persist_directory="chroma_word_store",
    embedding_function=embeddings
)

# Ajouter tous mes chunks dans le vector store
vector_store.add_documents(chunks)

print("Index vectoriel initalisé")

Index vectoriel initalisé


<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 3 — Créer le Retriever
</h2>


In [3]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 3})

In [4]:
# afficher un exemple de récupération
retriever.invoke("Quels sont les défis liés à l'extraction du texte Word ?")

[Document(id='f8c5412f-388b-45ee-bc65-643dcdd0efa1', metadata={'source': 'Document_RAG_Word.docx'}, page_content="2. Challenges rencontrés\nVoici les défis typiques rencontrés lors de l'extraction de texte Word :\n- Multiples sauts de ligne\n- Listes mal reconstruites\n- Tableaux transformés en texte\n- Titres non reconnus\n- Retours chariots inutiles\n3. Exemple de contenu complexe\nLe paragraphe suivant illustre un contenu souvent rencontré dans les entreprises. Il contient des descriptions opérationnelles, des éléments techniques et des blocs narratifs importants pour tester le comportement des extracteurs."),
 Document(id='a75dc956-b81f-4537-9c39-d23421875f39', metadata={'source': 'Document_RAG_Word.docx'}, page_content='1. Contexte général\nLes entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Définir le Prompt + le pipeline LCEL
</h2>


In [5]:
# Import
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert des systèmes RAG.
Réponds UNIQUEMENT à partir du contexte fourni ci-dessous.
Si l'information n'est pas présente, réponds simplement : "Ce n'est pas précisé dans le document."

Contexte :                                                                                           
{context}

Question :
{question} 
                                                                                                                                                                                                  
Réponse : 
""")

rag_chain = (
    {"context" : lambda x:retriever.invoke(x["question"]),
     "question" : lambda x: x["question"]}
    | prompt
    | llm
    | StrOutputParser()
)

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Execution : Poser une question sur le document Word
</h2>


<h3 style="color:#C0392B; font-weight:bold;">
   Ingestion Word -> nettoyage -> chunking -> indexation Chroma -> retriever -> pipeline LCEL


In [6]:
# 1 ) Définir la question

question = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)
question

"Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?"

In [7]:
# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question})

In [8]:
# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


QUESTION :
Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?

Réponse :
Les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG sont :
- Multiples sauts de ligne
- Listes mal reconstruites
- Tableaux transformés en texte
- Titres non reconnus
- Retours chariots inutiles


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 II. Ingestion Dynamique
</h1>

In [9]:
# 1) Etape 1 : Ingestion du nouveau document
from word_ingestion import WordIngestion

# ingestor personnalisé 
ingestor = WordIngestion(
    chunk_size= 500,
    chunk_overlap= 80
)

# Traiter un document word 
new_chunks = ingestor.process("../data/word_files/Document_RAG_Word_v2.docx")

print("Nombre de chunks :", len(chunks))

# Aperçu des premiers chunks 
for i, c in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print("=" *80)
    print(c.page_content[:500], "\n")


Nombre de chunks : 7
CHUNK 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG
Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel.
1. Contexte général 

CHUNK 2
1. Contexte général
Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers techniques. Ces documents doivent être ingérés, nettoyés, découpés en chunks et indexés pour permettre à un modèle de répondre de façon cohérente aux requêtes des utilisateurs.
2. Challenges rencontrés 

CHUNK 

In [10]:
# Etape 2 - Enrichir les métadonnées + ajout à Chroma

#1) Ajouter une métadonnée 'source' pour identifier l'origine
for c in new_chunks :
    c.metadata["source"] = "Document_RAG_Word_v2.docx"

#2) Ajouter les nouveaux chunks au vector store existant
vector_store.add_documents(new_chunks)

print("Nouveaux documents ajoutés au vector store")
print(f"Nombre total de documents indexés : {vector_store._collection.count()}")

Nouveaux documents ajoutés au vector store
Nombre total de documents indexés : 14


In [11]:
# Etape 3 : Vérifier que le retriever "voit" les nouveaux documents

# 1 ) Définir la question

question_new = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)

# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question_new})

# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


QUESTION :
Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?

Réponse :
Les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG sont :

1. Supprimer les espaces inutiles
2. Problème d’affichage
3. Préparer le pipeline
4. Okandata


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 III. Techniques avancées RAG : Mémoire conversationnelle + Vector Store
</h1>

In [12]:
# =======================================
#  Etape 1 — IMPORTS & FORMATTEUR DOCS
# =======================================

from operator import itemgetter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableWithMessageHistory
)
from langchain_core.output_parsers import StrOutputParser

from langchain_community.chat_message_histories import ChatMessageHistory

from pdf_ingestion import PDFIngestion

# Fonction obligatoire pour éviter l'erreur TypeError (liste -> texte)
def format_docs(docs) :
    return "\n\n---\n\n".join(d.page_content for d in docs)

format_docs_runnable = RunnableLambda(format_docs)


In [13]:
# =======================================
#  Etape 2 — INGESTION PDF 
# =======================================

ingestor = PDFIngestion(chunk_size=1200, chunk_overlap=150) 

chunks_pdf = ingestor.process("../data/pdf_files/rapport-activite_credit_mutuelle.pdf")

print("Nombre de chunks chargés :", len(chunks_pdf))
print("Aperçu du premier chunk :", chunks_pdf[4].page_content[:500])

Nombre de chunks chargés : 208
Aperçu du premier chunk : Le modèle mutualiste reste-t-il un atout et un levier pour
le développement présent et futur du Groupe ?
Daniel Baal : Oui.et.plus.que.jamais.!.Il.repose.sur.des.
valeurs.de.proximité,.de.solidarité.et.de.gouvernance.
démocratique,. qui. font. notre. singularité. et. notre.
force. face. aux. défis. économiques. et. sociétaux .
actuels.. Dans. un. environnement. en. constante .
évolution,.marqué.par.la.digitalisation,.la.transition.
écologique.et.les.attentes.croissantes.des.clients.en.
matière.d


In [14]:
# =======================================
#  Etape 3 — INDEXATION DANS CHROMA
# =======================================

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

vector_store_pdf = Chroma(
    collection_name="rapport_mutuelle",
    persist_directory= "chroma_pdf_store",
    embedding_function=embeddings
)

# vector_store_pdf.delete_collection()

vector_store_pdf.add_documents(chunks_pdf)

print("Index PDF en base :", vector_store_pdf._collection.count())

Index PDF en base : 208


In [15]:
# =======================================
#  Etape 4 — RETRIEVER
# =======================================

retriever_pdf = vector_store_pdf.as_retriever(search_kwargs = {"k":3})
retriever_pdf

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x75e237c29610>, search_kwargs={'k': 3})

In [16]:
# =======================================
#  Etape 5 — PROMPT AVEC HISTORIQUE
# =======================================

prompt_conv = ChatPromptTemplate.from_messages([
    ("system",
    "Tu es un assistant expert qui analyse le rapport d'activité du Crédit Mutuel."
    "Tu t'appuis uniquement sur les extraits fournis. Réponds clairement et en français"),

    MessagesPlaceholder("history"),

    ("human",
    "Voici les extraits pertinents du rapport : \n\n{context}\n\n"
    "Ma question : {question}")
])


In [17]:
# =======================================
#  Etape 6 — CHAÎNE RAG LCEL
# =======================================

llm = ChatOpenAI(model = "gpt-4o-mini")

rag_chain = (
    {
        "question" : itemgetter("question"),
        "context" : itemgetter("question") | retriever_pdf | format_docs_runnable,
        "history" : itemgetter("history"),   
    }
    | prompt_conv
    | llm
    | StrOutputParser()
)


In [18]:
# =======================================
#  Etape 7 — MEMORIRE CONVERSATIONNELLE
# =======================================

store = {}

def get_session_history(session_id : str) : 
    if session_id not in store :
        store[session_id] = ChatMessageHistory()
    return store[session_id]

rag_with_memory = RunnableWithMessageHistory(


    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)



/home/lady/Documents/insta/ia-generative/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
# =======================================
#  Etape 8 — TEST DU RAG CONVERSATIONNELLE
# =======================================

session_id = "mutuelle_session_1"

def ask(question) :
    response = rag_with_memory.invoke(
        {"question": question},
        config = {"configurable":{"session_id" : session_id}}
    )

    print(f"\n ? {question} \n")
    print("Réponse :\n", response)
    print("\n" + "="*80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l'accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")
ask("Et résume moi tout cela en une phrase comme si tu parlais à un sociétaire")

/tmp/ipykernel_35090/1724105824.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = ChatMessageHistory()



 ? Quels sont les grands axes stratégiques du rapport d'activité ? 

Réponse :
 Les grands axes stratégiques du rapport d'activité du Crédit Mutuel sont les suivants :

1. **Engagement envers les communautés** : Cela inclut des initiatives de mécénat et des actions de développement des territoires. Le groupe vise à renforcer son impact positif sur les différentes communautés.

2. **Innovation et solidarité** : Le rapport mentionne des outils comme le Dividende Sociétal et le Fonds de dotation pour promouvoir l'innovation tout en soutenant des causes solidaires.

3. **Développement local** : À travers les stratégies des groupes régionaux et les engagements des sociétés à mission, le Crédit Mutuel contribue au développement des territoires.

4. **Relation client et qualité de service** : Le groupe se positionne comme numéro un en termes de relation client, avec un engagement reconnu pour la qualité de service.

5. **Protection des clients et prévention de la précarité** : Des mécanismes

In [20]:
# =======================================
# 📌 Cellule 8 — TESTS DU RAG
# =======================================

session_id = "mutuelle-session-1"

def ask(question):
    response = rag_with_memory.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"\n❓ {question}\n")
    print("💬 Réponse :\n", response)
    print("\n" + "="*80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l’accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")



❓ Quels sont les grands axes stratégiques du rapport d'activité ?

💬 Réponse :
 Les grands axes stratégiques du rapport d'activité du Crédit Mutuel se déclinent comme suit :

1. **Engagement envers les communautés** : Cela inclut le mécénat et le développement des territoires, soulignant l'importance de l'implication locale.

2. **Promotion de l'innovation et de la solidarité** : Le développement de nouveaux outils, comme le Dividende Sociétal et le Fonds de dotation, est mis en avant pour favoriser l'innovation et soutenir les initiatives solidaires.

3. **Soutien au développement local** : Les groupes régionaux ont des stratégies axées sur leur contribution au développement local, en se fixant des engagements en tant que sociétés à mission.

4. **Protection des clients et gestion des risques** : Un cadre est établi pour protéger les informations des clients, anticiper et accompagner les situations de vulnérabilité, ainsi que gérer les risques liés à la fraude et à la corruption.

5.

In [21]:
ask("Et résume-moi tout cela en une phrase, comme si tu parlais à un sociétaire.")



❓ Et résume-moi tout cela en une phrase, comme si tu parlais à un sociétaire.

💬 Réponse :
 Cher sociétaire, au Crédit Mutuel, votre voix compte véritablement, car nous valorisons la solidarité, l'égalité et la responsabilité sociale à travers notre fonctionnement coopératif, vous permettant ainsi de participer activement aux décisions locales tout en bénéficiant d'un service de proximité.



In [22]:
ask("Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?")


❓ Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?

💬 Réponse :
 Les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG (Retrieval-Augmented Generation) incluent :

1. **Variabilité des formats** : Les documents Word peuvent contenir une grande variété de mises en page, de polices, d'images et de tableaux, rendant difficile une extraction de texte uniforme et structurée.

2. **Qualité du texte extrait** : Assurer que le texte extrait est cohérent et exempt d'erreurs est crucial, surtout si des informations importantes ou sensibles sont présentes. Les imperfections dans l'extraction peuvent nuire à la qualité des résultats générés par le pipeline RAG.

3. **Gestion des métadonnées** : Les documents peuvent contenir des métadonnées importantes qui ne sont pas visibles en tant que texte. L'extraction de ces données peut être nécessaire pour enrichir le contexte ou la pertinence des réponses fou

In [23]:
# tester la memoire conversationnelle en posant une question sur un document Word
ask("qui est ISABELLE FERRAND ?")


❓ qui est ISABELLE FERRAND ?

💬 Réponse :
 Isabelle Ferrand est la Directrice générale de la Confédération Nationale du Crédit Mutuel. Elle occupe un poste clé au sein de l'organisation, contribuant à la direction et à la gestion des opérations du groupe bancaire.



In [24]:
ask("Quelle est la position d'ISABELLE FERRAND ?")


❓ Quelle est la position d'ISABELLE FERRAND ?

💬 Réponse :
 Isabelle Ferrand occupe le poste de Directrice générale de la Confédération Nationale du Crédit Mutuel.

